# Import Dependencies

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import logging
import json
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')
from lime.lime_text import LimeTextExplainer


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

c:\Users\User\miniconda3\envs\py39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



PyTorch version: 2.8.0+cpu
CUDA available: False
Device: CPU


# Configs

In [3]:
# File paths
CSV_PATH = 'litigation_data/litigation.csv'  # Your litigation CSV file
OUTPUT_DIR = 'litigation_output'

# Model selection: 'finbert', 'bert', or 'electra'
MODEL_TYPE = 'finbert'

# Training parameters
SAMPLE_SIZE = 100      # Number of cases to use (set to None for full dataset)
EPOCHS = 3             # Training epochs
BATCH_SIZE = 8         # Smaller batch size for limited data
LEARNING_RATE = 2e-5
MAX_LENGTH = 512       # Max token length

# Create output directory
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  Model: {MODEL_TYPE.upper()}")
print(f"  Sample size: {SAMPLE_SIZE if SAMPLE_SIZE else 'Full dataset'}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")

Configuration:
  Model: FINBERT
  Sample size: 100
  Epochs: 3
  Batch size: 8


# Init Data Loader

In [4]:
class LitigationDataset(Dataset):
    """PyTorch Dataset for litigation text classification"""
    
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("✓ Dataset class defined")

✓ Dataset class defined


# Load Data 

In [5]:
# Model configurations
MODELS = {
    'finbert': 'ProsusAI/finbert',
    'bert': 'bert-base-uncased',
    'electra': 'google/electra-base-discriminator'
}

# Insider trading keywords for auto-labeling
INSIDER_KEYWORDS = [
    'insider trading', 'material nonpublic', 'tipping', 'tippee',
    'rule 10b-5', 'section 10(b)', 'misappropriation', 'securities fraud'
]

def has_insider_keywords(text):
    """Check if text contains insider trading keywords"""
    text_lower = str(text).lower()
    return int(any(kw in text_lower for kw in INSIDER_KEYWORDS))

def extract_company(text):
    """Simple company name extraction"""
    match = re.search(r'(?:SEC charges|charges against)\s+([A-Z][A-Za-z\s&]+?)(?:\s+(?:and|with|for))', str(text))
    if match:
        return match.group(1).strip()
    match = re.search(r'^([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+){0,2})', str(text))
    return match.group(1).strip() if match else "Unknown"

# Load data
print(f"Loading data from {CSV_PATH}...")
df_full = pd.read_csv(CSV_PATH, index_col=0)

# Combine title + text
df_full['text'] = (df_full['title'].fillna('') + '. ' + df_full['lt'].fillna('')).str.strip()
df_full['text'] = df_full['text'].apply(lambda x: re.sub(r'\s+', ' ', x))

# Auto-label based on keywords
df_full['label'] = df_full['text'].apply(has_insider_keywords)

# Extract company names
df_full['company'] = df_full['text'].apply(extract_company)

print(f"\nFull dataset: {len(df_full)} cases")
print(f"  Insider trading: {df_full['label'].sum()} ({df_full['label'].mean()*100:.1f}%)")
print(f"  Years: {df_full['yr'].min()} - {df_full['yr'].max()}")

# Sample data if specified
if SAMPLE_SIZE and SAMPLE_SIZE < len(df_full):
    # Stratified sampling to maintain label distribution
    df = df_full.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), SAMPLE_SIZE // 2), random_state=42)
    ).reset_index(drop=True)
    
    print(f"\n✓ Sampled to {len(df)} cases (stratified)")
    print(f"  Insider trading: {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
else:
    df = df_full.copy()
    print(f"\n✓ Using full dataset")

# Display sample
print("\nSample cases:")
print(df[['lt_no', 'yr', 'company', 'label']].head(10))

Loading data from litigation_data/litigation.csv...

Full dataset: 7988 cases
  Insider trading: 5349 (67.0%)
  Years: 1996 - 2017

✓ Sampled to 100 cases (stratified)
  Insider trading: 50 (50.0%)

Sample cases:
   lt_no    yr                 company  label
0  19103  2005                 Unknown      0
1  18621  2004           Rollin S Dick      0
2  23808  2017  SEC Charges Investment      0
3  20471  2008                 Unknown      0
4  17381  2002                 Unknown      0
5  20363  2007                 Unknown      0
6  20763  2008                 Unknown      0
7  18665  2004                 Unknown      0
8  23705  2016                 Unknown      0
9  20808  2008                 Unknown      0


# Train/Test/Val Split

In [6]:
# Initialize tokenizer
model_name = MODELS[MODEL_TYPE]
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Split data: 70% train, 20% test, 10% val
train_val, test = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train, val = train_test_split(train_val, test_size=0.125, stratify=train_val['label'], random_state=42)  # 0.125 * 0.8 = 0.1

print(f"\nData splits:")
print(f"  Train: {len(train)} ({len(train)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val)} ({len(val)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test)} ({len(test)/len(df)*100:.1f}%)")

# Create PyTorch datasets
train_dataset = LitigationDataset(train['text'].tolist(), train['label'].tolist(), tokenizer, MAX_LENGTH)
val_dataset = LitigationDataset(val['text'].tolist(), val['label'].tolist(), tokenizer, MAX_LENGTH)
test_dataset = LitigationDataset(test['text'].tolist(), test['label'].tolist(), tokenizer, MAX_LENGTH)

print("\n✓ Datasets created")

Loading tokenizer for ProsusAI/finbert...

Data splits:
  Train: 70 (70.0%)
  Val:   10 (10.0%)
  Test:  20 (20.0%)

✓ Datasets created


# Train FinBert

In [8]:
def compute_metrics(pred):
    """Compute evaluation metrics"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading {MODEL_TYPE.upper()} model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2, 
    ignore_mismatched_sizes=True
).to(device)

# Training arguments
training_args = TrainingArguments(
    output_dir=f'{OUTPUT_DIR}/training',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=10,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch'
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train
print(f"\nTraining {MODEL_TYPE.upper()} for {EPOCHS} epochs...")
print("=" * 70)
trainer.train()

# Save model
model_path = Path(OUTPUT_DIR) / 'model'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"\n✓ Model saved to {model_path}")

Loading FINBERT model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training FINBERT for 3 epochs...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.712786,0.500000,0.500000,1.000000,0.666667
2,0.676800,0.704133,0.700000,0.625000,1.000000,0.769231
3,0.578700,0.701884,0.600000,0.571429,0.800000,0.666667



✓ Model saved to litigation_output\model


# Eval Results

In [9]:
model.eval()
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_preds, all_labels, all_probs = [], [], []

print("Evaluating on test set...")
with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        probs = torch.softmax(outputs.logits, dim=1)
        preds = probs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

# Calculate metrics
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
accuracy = accuracy_score(all_labels, all_preds)

print("\n" + "=" * 70)
print(f"{MODEL_TYPE.upper()} Test Set Results")
print("=" * 70)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Not Insider', 'Insider Trading']))

# Store results
test_results = test.copy()
test_results['predicted'] = all_preds
test_results['probability'] = all_probs

Evaluating on test set...



FINBERT Test Set Results
Accuracy:  0.6000
Precision: 0.5833
Recall:    0.7000
F1-Score:  0.6364

Classification Report:
                 precision    recall  f1-score   support

    Not Insider       0.62      0.50      0.56        10
Insider Trading       0.58      0.70      0.64        10

       accuracy                           0.60        20
      macro avg       0.60      0.60      0.60        20
   weighted avg       0.60      0.60      0.60        20



# Full Dataset Results

In [10]:
# Predict on entire dataset
full_dataset = LitigationDataset(df['text'].tolist(), [0]*len(df), tokenizer, MAX_LENGTH)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_preds_full, all_probs_full = [], []

print(f"Generating predictions for all {len(df)} cases...")
with torch.no_grad():
    for batch in full_loader:
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        probs = torch.softmax(outputs.logits, dim=1)
        all_preds_full.extend(probs.argmax(dim=1).cpu().numpy())
        all_probs_full.extend(probs[:, 1].cpu().numpy())

# Add predictions to dataframe
df_classified = df.copy()
df_classified[f'{MODEL_TYPE}_pred'] = all_preds_full
df_classified[f'{MODEL_TYPE}_prob'] = all_probs_full
df_classified[f'{MODEL_TYPE}_class'] = df_classified[f'{MODEL_TYPE}_prob'].apply(
    lambda x: 'Definite Insider' if x >= 0.85 else ('Possible Insider' if x >= 0.50 else 'Not Insider')
)

print("\n✓ Predictions complete")

# Summary
print(f"\nClassification Summary:")
summary = df_classified[f'{MODEL_TYPE}_class'].value_counts()
for category, count in summary.items():
    print(f"  {category}: {count} ({count/len(df_classified)*100:.1f}%)")

Generating predictions for all 100 cases...

✓ Predictions complete

Classification Summary:
  Possible Insider: 62 (62.0%)
  Not Insider: 38 (38.0%)


# LIME For Interperobility 

In [11]:
# Quick explanation using transformers-interpret (lighter than LIME)
from transformers_interpret import SequenceClassificationExplainer

cls_explainer = SequenceClassificationExplainer(model, tokenizer)

# Explain first case
case = df_classified.iloc[0]
text = case['text'][:512]  # Truncate to avoid memory issues

word_attributions = cls_explainer(text, class_name='LABEL_1')

print(f"\nCase: {case['lt_no']} | Probability: {case[f'{MODEL_TYPE}_prob']:.3f}")
print("\nTop 10 words influencing prediction:")
sorted_attrs = sorted(word_attributions, key=lambda x: abs(x[1]), reverse=True)[:10]

for word, attribution in sorted_attrs:
    direction = "→ Insider" if attribution > 0 else "→ Not Insider"
    print(f"  {word:20s}: {attribution:+.4f} {direction}")


Case: 19103 | Probability: 0.426

Top 10 words influencing prediction:
  ##d                 : -0.4322 → Not Insider
  violated            : +0.4204 → Insider
  filed               : +0.3319 → Insider
  alleging            : +0.2936 → Insider
  that                : -0.2599 → Not Insider
  against             : +0.1997 → Insider
  it                  : -0.1918 → Not Insider
  two                 : +0.1562 → Insider
  the                 : -0.1537 → Not Insider
  james               : +0.1339 → Insider
